In [ ]:
import openeo
from data_preparation.parcel_stats.openeo import OpenEOZonalStats
from data_preparation.parcel_stats.multiuser import (
    load_openeo_users_from_db,
)
from data_preparation.parcel_stats.core.base import ParcelStatsBase

In [ ]:
users_list = load_openeo_users_from_db()
print([username for username, _ in users_list])

In [ ]:
from openeo.rest import OpenEoApiError

OPENEO_URL = OpenEOZonalStats.OPENEO_URL
TERMINAL_STATES = {"finished", "error", "failed", "canceled"}

for username, password in users_list:
    connection = openeo.connect(OPENEO_URL, auto_validate=False)
    connection.authenticate_oidc_resource_owner_password_credentials(
        username=username,
        password=password,
        client_id=ParcelStatsBase.OPENEO_OIDC_PASSWORD_CLIENT_ID,
    )

    jobs = connection.list_jobs() or []
    print(50*"=")
    print(username)
    print(f"Jobs: {jobs}")

    # Reverse the list with jobs in place to delete the most old ones first
    jobs.reverse()

    deleted_count = 0
    not_found_count = 0

    for job_info in jobs:
        job_id = job_info.get("id")
        state = (job_info.get("status") or "").lower()

        if not job_id or state not in TERMINAL_STATES:
            continue

        try:
            connection.job(job_id).delete()
            deleted_count += 1
        except OpenEoApiError as e:
            if e.http_status_code == 404:
                # Job disappeared between list and delete (race condition)
                not_found_count += 1
                print(f"Job {job_id} not found (already deleted?)")
            else:
                raise

    print(
        f"Deleted {deleted_count} terminal jobs (states: {', '.join(sorted(TERMINAL_STATES))}, already gone: {not_found_count})"
    )